# 06 - NLI Contradiction Judgment, Chunk-Level (ContraDoc)

Score every candidate chunk pair from step 05 with a pretrained NLI model. The candidate record already carries `chunk_a.source_text` and `chunk_b.source_text`, so there's no triple-pair-to-sentence lookup and no dedup (chunk pairs are unique by construction under the new schema).

**Input:** `data/processed/ContraDoc/chunk_candidates.jsonl`
**Output:** `data/processed/ContraDoc/nli_predictions.jsonl`

Model: `cross-encoder/nli-deberta-v3-base` (off-the-shelf; labels `[contradiction, entailment, neutral]` after softmax).

In [1]:
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from sentence_transformers import CrossEncoder

INPUT_PATH = Path("data/processed/ContraDoc/chunk_candidates.jsonl")
OUTPUT_PATH = Path("data/processed/ContraDoc/nli_predictions.jsonl")
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
BATCH_SIZE = 32

## Load chunk-pair candidates

In [2]:
candidates = [json.loads(line) for line in INPUT_PATH.open(encoding="utf-8")]
print(f"Loaded {len(candidates)} candidate chunk pairs")
print(f"  is_gold_pair=True: {sum(1 for c in candidates if c['is_gold_pair'])}")
src = defaultdict(int)
for c in candidates:
    src[c["source"]] += 1
for k, v in src.items():
    print(f"  source={k}: {v}")

Loaded 2451 candidate chunk pairs
  is_gold_pair=True: 23
  source=struct: 451
  source=vector: 1886
  source=struct+vector: 114


## Load NLI model + sanity check

In [3]:
model = CrossEncoder(NLI_MODEL)
id2label = model.config.id2label
label2id = {v.lower(): int(k) for k, v in id2label.items()}
CONTRA_IDX = label2id["contradiction"]
ENTAIL_IDX = label2id["entailment"]
NEUTRAL_IDX = label2id["neutral"]
print(f"id2label = {id2label}  (contra_idx={CONTRA_IDX})")

test = model.predict(
    [
        ("He donated his kidney.", "He never donated his kidney."),
        ("The cat is sleeping.", "The cat is asleep."),
    ],
    apply_softmax=True,
)
for (p, h), s in zip([("donated", "never donated"), ("sleeping", "asleep")], test):
    print(f"  {p} / {h} -> {id2label[int(np.argmax(s))]}  contra={s[CONTRA_IDX]:.2f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


id2label = {0: 'contradiction', 1: 'entailment', 2: 'neutral'}  (contra_idx=0)
  donated / never donated -> contradiction  contra=1.00
  sleeping / asleep -> entailment  contra=0.00


## Run NLI on every candidate chunk pair

No dedup needed — chunk pairs are already unique under the new schema.

In [4]:
pairs = [(c["chunk_a"]["source_text"], c["chunk_b"]["source_text"]) for c in candidates]
scores = model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=True, apply_softmax=True)
scores = np.asarray(scores)
print(f"Scored {len(scores)} pairs. Shape: {scores.shape}")

for c, s in zip(candidates, scores):
    c["nli_contradiction"] = float(s[CONTRA_IDX])
    c["nli_entailment"] = float(s[ENTAIL_IDX])
    c["nli_neutral"] = float(s[NEUTRAL_IDX])
    c["nli_pred"] = id2label[int(np.argmax(s))].lower()

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Scored 2451 pairs. Shape: (2451, 3)


## Evaluation across contradiction-probability thresholds

In [5]:
is_gold = np.array([c["is_gold_pair"] for c in candidates], dtype=bool)
contra = np.array([c["nli_contradiction"] for c in candidates])
doc_ids = [c["doc_id"] for c in candidates]

n_total_gold = int(is_gold.sum())
gold_doc_ids = {doc_ids[i] for i in range(len(candidates)) if is_gold[i]}
n_gold_docs = len(gold_doc_ids)

print(f"Gold chunk pairs in candidate set: {n_total_gold}  across {n_gold_docs} docs")
print()
header = f"{'thresh':>6}  {'#pred':>6}  {'TP':>4}  {'FP':>5}  {'FN':>4}  {'Prec':>6}  {'Pair-R':>6}  {'F1':>6}  {'Doc-R':>11}"
print(header)
print("-" * len(header))
for t in [0.3, 0.5, 0.7, 0.9, 0.95]:
    pred = contra >= t
    tp = int((pred & is_gold).sum())
    fp = int((pred & ~is_gold).sum())
    fn = int((~pred & is_gold).sum())
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    doc_with_tp = {doc_ids[i] for i in np.where(pred & is_gold)[0]}
    doc_r = len(doc_with_tp) / max(n_gold_docs, 1)
    print(
        f"{t:>6.2f}  {int(pred.sum()):>6}  {tp:>4}  {fp:>5}  {fn:>4}  "
        f"{prec:>5.1%}  {rec:>5.1%}  {f1:>5.1%}  "
        f"{len(doc_with_tp):>2}/{n_gold_docs:<2} {doc_r:>5.1%}"
    )

Gold chunk pairs in candidate set: 23  across 23 docs

thresh   #pred    TP     FP    FN    Prec  Pair-R      F1        Doc-R
----------------------------------------------------------------------
  0.30     612    22    590     1   3.6%  95.7%   6.9%  22/23 95.7%
  0.50     581    22    559     1   3.8%  95.7%   7.3%  22/23 95.7%
  0.70     552    22    530     1   4.0%  95.7%   7.7%  22/23 95.7%
  0.90     514    22    492     1   4.3%  95.7%   8.2%  22/23 95.7%
  0.95     490    22    468     1   4.5%  95.7%   8.6%  22/23 95.7%


## Save predictions + show samples

In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for c in candidates:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"Saved {len(candidates)} predictions -> {OUTPUT_PATH.resolve()}")

tp_idx = [i for i in np.argsort(-contra) if is_gold[i]][:3]
print()
print("Top 3 highest-contradiction *gold* chunk pairs:")
for i in tp_idx:
    c = candidates[i]
    print(f"  doc={c['doc_id']}  contra={c['nli_contradiction']:.3f}")
    print(f"    A: {c['chunk_a']['source_text'][:140]}")
    print(f"    B: {c['chunk_b']['source_text'][:140]}")

fp_idx = [i for i in np.argsort(-contra) if not is_gold[i]][:3]
print()
print("Top 3 highest-contradiction *non-gold* chunk pairs:")
for i in fp_idx:
    c = candidates[i]
    print(f"  doc={c['doc_id']}  contra={c['nli_contradiction']:.3f}")
    print(f"    A: {c['chunk_a']['source_text'][:140]}")
    print(f"    B: {c['chunk_b']['source_text'][:140]}")

Saved 2451 predictions -> D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\nli_predictions.jsonl

Top 3 highest-contradiction *gold* chunk pairs:
  doc=3502252168_1  contra=1.000
    A: Capel Lligwy ( sometimes referred to as Hen Gapel Lligwy ) is a ruined chapel near Rhos Lligwy in Anglesey , north Wales , dating back to th
    B: Capel Lligwy is a well-preserved chapel near Rhos Lligwy in Anglesey, north Wales, dating back to the first half of the 12th century.
  doc=3499318683_3  contra=1.000
    A: Both Sébergué and Ndikert ranked first in their respective heats and advanced past the qualification round.
    B: Both Sébergué and Ndikert ranked seventh in their respective heats and did not advance past the qualification round.
  doc=3489738256_3  contra=1.000
    A: The LHC generates up to 600 million particles per second, with a beam circulating for 10 hours, traveling more than 6 billion miles (more th
    B: The LHC generates only 100 part

## Per-type recall breakdown (NLI at each threshold)

How well does NLI catch each ContraDoc contradiction type? Multi-label docs contribute to every listed type.

In [7]:
from collections import defaultdict

# Load contradiction types per doc from triples.jsonl
doc_types = {}
with Path("data/processed/ContraDoc/triples.jsonl").open(encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["contradiction"] == "YES":
            doc_types[r["doc_id"]] = [t for t in (r.get("contra_type") or "none").split("|") if t]


def cand_key(c):
    return tuple(sorted([(c["doc_id"], c["chunk_a"]["sentence_id"]), (c["doc_id"], c["chunk_b"]["sentence_id"])]))


# Gold chunk pairs represented as (doc_id, sid) tuple pairs
gold_set = {cand_key(c) for c in candidates if c["is_gold_pair"]}


def per_type_recall(pairs, name):
    type_totals = defaultdict(int)
    type_caught = defaultdict(int)
    for p in gold_set:
        doc_id = p[0][0]
        for t in doc_types.get(doc_id, ["unknown"]):
            type_totals[t] += 1
            if p in pairs:
                type_caught[t] += 1
    all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
    print(f"\n{name}:")
    print(f"  {'type':30s}  caught  total  recall")
    print("  " + "-" * 52)
    for t in all_types:
        caught, total = type_caught[t], type_totals[t]
        print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")


for thr in [0.30, 0.50, 0.70, 0.90, 0.95]:
    flagged = {cand_key(c) for c in candidates if c["nli_contradiction"] >= thr}
    per_type_recall(flagged, f"NLI @ {thr:.2f}")


NLI @ 0.30:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             15     15  100.0%
  Numeric                              7      7  100.0%
  Perspective/View/Opinion             4      4  100.0%
  Negation                             3      3  100.0%
  Emotion/Mood/Feeling                 2      2  100.0%
  Factual                              2      2  100.0%
  Relation                             0      1    0.0%

NLI @ 0.50:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             15     15  100.0%
  Numeric                              7      7  100.0%
  Perspective/View/Opinion             4      4  100.0%
  Negation                             3      3  100.0%
  Emotion/Mood/Feeling                 2      2  100.0%
  Factual                              2      2  100.0%
  Relation              